# TANGLE at a glance

TANGLE builds a fiber network by approximating manufacturing as an
ordered, quasi-static recipe. Python describes the geometry and recipe;
Rust validates and packs them; CubeCL keeps the active world resident on
a CPU or GPU while contact, mechanics, and topology evolve.

```text
Cell + materials + fiber collections
                 ↓
      ordered manufacturing recipe
                 ↓
  CubeCL relaxation + adaptive topology
                 ↓
     RunResult → OVITO / BPM
```

This deliberately small two-ply example introduces the ideas developed
individually in tutorials 02–14. All physical quantities use SI units.
Building the objects is cheap; the guarded execution cell is the only
part that launches the solver.

In [ ]:
import inspect
from pathlib import Path

import tangle

# Catch a stale compiled extension before a long recipe reaches a
# keyword added by a newer notebook.
run_parameters = inspect.signature(tangle.Recipe.run).parameters
if "debug_ovito_path" not in run_parameters:
    raise RuntimeError(
        "This kernel has an older compiled TANGLE extension loaded. "
        "Run the installation cell in tutorial 00, restart the kernel, "
        "select Python (TANGLE), and then run this notebook from the top."
    )

# Recipe construction is cheap; this switch controls the actual solve
# and all filesystem output.
RUN_OVERVIEW = False
output = Path("output/overview")

## 1. Describe the domain and fibers

The cell is periodic in-plane and bounded through-thickness. A material
supplies capsule diameter and an optional admissible bend radius. A
collection holds placed centerlines, preferred rest centerlines, and
metadata before anything enters the simulation.

Here the top large fiber is placed slightly curved but has a straight
rest centerline: relaxation treats that initial curvature as bending.
Bulk and biased collections can instead be made with TANGLE's seeded
fiber generators.

In [ ]:
# Axis order is x, y, z. Periodic x/y represent a repeating sheet;
# bounded z retains physical top and bottom surfaces.
cell = tangle.Cell(
    [1.0e-3, 1.0e-3, 1.5e-3],
    periodic=[True, True, False],
)
# Diameter controls contact geometry. minimum_bend_radius is the hard
# admissible-curvature scale, not the preferred rest shape.
small = tangle.Material(
    "small", diameter=7.0e-6, minimum_bend_radius=35.0e-6
)
large = tangle.Material(
    "large", diameter=19.0e-6, minimum_bend_radius=75.0e-6
)

# Collections are detached local geometry. formation_layer metadata
# lets later recipe operations address manufacturing plies.
bottom = tangle.FiberCollection("bottom ply")
bottom.add_fiber(
    [[-0.35e-3, -0.10e-3, 0.0], [0.35e-3, -0.10e-3, 0.0]],
    small,
    tags={"family": "x"},
    formation_layer=0,
)
bottom.add_fiber(
    [[0.12e-3, -0.35e-3, 0.0], [0.12e-3, 0.35e-3, 0.0]],
    small,
    tags={"family": "y"},
    formation_layer=0,
)

top = tangle.FiberCollection("top ply")
# `placed` is the literal initial geometry. Giving it a straight rest
# shape means the visible waviness initially stores bending strain.
placed = [
    [-0.35e-3, 0.0, 0.0],
    [0.0, 12.0e-6, 0.0],
    [0.35e-3, 0.0, 0.0],
]
straight_rest = [
    [-0.35e-3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.35e-3, 0.0, 0.0],
]
top.add_fiber(
    placed,
    large,
    rest_centerline=straight_rest,
    tags={"family": "needle-eligible"},
    formation_layer=1,
)
top.add_fiber(
    [[-0.08e-3, -0.35e-3, 0.0], [-0.08e-3, 0.35e-3, 0.0]],
    small,
    tags={"family": "y"},
    formation_layer=1,
)

print(len(bottom), len(top), bottom.layers(), top.layers())

## 2. Write the manufacturing recipe

A recipe is an ordered operation list, not a time integrator. It can
insert dormant collections, establish temporary kinematic targets,
relax, release targets, compact the cell, and capture persistent
junction topology. Explicit relaxation gates make the intended sequence
auditable before the expensive run begins.

This example settles the bottom ply, inserts and lowers the top ply,
performs one visible needling displacement, releases the needle,
compacts through-thickness, performs a strict final relaxation, and
finally records selected contacts as junctions.

In [ ]:
# Axis indices are 0=x, 1=y, 2=z. Here z is the stacking or
# through-thickness direction used by all layer-aware operations.
recipe = tangle.Recipe(cell, layer_axis=2)

# Insert and settle the first ply before activating the second.
recipe.insert(bottom, translation=[0.5e-3, 0.5e-3, 0.35e-3])
recipe.relax(maximum_iterations=2_000)

recipe.insert(top, translation=[0.5e-3, 0.5e-3, 0.80e-3])
recipe.place_layer_above(
    layer=1, gap=5.0e-6, stiffness=0.5, max_translation=5.0e-6
)

# Penetration is hard during assembly, while bend cleanup is temporarily
# soft so manufacturing motion can finish before the final strict pass.
assembly_policy = tangle.SolvePolicy(
    "contact-first assembly",
    solver_penetration=0.2e-6,
    solver_curvature_ratio=2.0,
    acceptance_penetration=0.3e-6,
    penetration_enforcement="hard",
    acceptance_curvature_ratio=2.0,
    curvature_enforcement="soft",
    maximum_iterations=4_000,
    on_exhaustion="reject",
)
assembly_overrides = tangle.RelaxationOverrides()
assembly_overrides.bend_stiffness = 0.2
recipe.relax_with_policy(assembly_policy, assembly_overrides)
recipe.release_layer_targets()

# The needle pulls eligible large-fiber vertices; neighboring fibers
# move only when contact transmits that displacement.
recipe.needle_layer_circular(
    layer=1,
    center=[0.5e-3, 0.5e-3],
    diameter=120.0e-6,
    depth=0.25e-3,
    minimum_fiber_diameter=15.0e-6,
    stiffness=0.75,
    max_translation=5.0e-6,
    maximum_translation_over_fiber_diameter=0.5,
)
recipe.relax_until_targets_reached(0.2e-6, 3_000)
recipe.release_needles()

# Only z may shrink because the axis weights are [x=0, y=0, z=1].
compaction = tangle.CompactionSettings.volume_fraction(
    0.002, axis_weights=[0.0, 0.0, 1.0]
)
compaction.maximum_steps = 20
compaction.maximum_relax_windows = 4
recipe.compact(compaction)
recipe.relax(maximum_iterations=4_000)

# Ordinary contacts remain transient until this explicit late capture.
junctions = tangle.JunctionPolicy("late contact capture", "bond")
junctions.maximum_surface_gap = 0.2e-6
junctions.probability = 1.0
junctions.material_pairs = [("large", "small")]
recipe.capture_junctions(junctions)

## 3. Configure one resident solve

Global relaxation settings control contact resolution, fiber mechanics,
batching, backend selection, and adaptive refinement/coarsening. Recipe
policies above may temporarily override a few of them for one stage.
A checkpoint can preserve the resident formation state and operation
cursor for continuation.

The CPU backend executes the same CubeCL kernels without a GPU. Change
`backend` to `"wgpu"` for a supported accelerator.

In [ ]:
# These are global defaults; a recipe policy may temporarily override
# selected values for one manufacturing stage.
settings = tangle.RelaxationSettings()
settings.backend = "cpu"
settings.motion_model = "flexible"
settings.penetration_tolerance = 0.1e-6
settings.curvature_ratio_tolerance = 1.05
settings.max_step = 2.0e-6
# Manufacturing targets are updated between batches. A short batch
# keeps this small target-heavy recipe responsive without controlling
# how many OVITO frames are retained.
settings.iterations_per_batch = 6
settings.adaptive_segmentation = (
    tangle.AdaptiveSegmentationSettings.profile("balanced")
)

# The restart stores the recipe cursor and resident solver state, not
# merely a final list of downloaded centerlines.
checkpoint = tangle.CheckpointSettings(
    "tutorial-overview",
    output / "overview.restart",
    interval_iterations=500,
)

print("Recipe operations:")
for number, operation in enumerate(recipe.operations(), start=1):
    print(f"{number:2d}. {operation}")

## 4. Run, inspect, and export

`Recipe.run()` uploads the packed world, executes the ordered recipe,
and returns final geometry plus convergence, topology, transfer, event,
checkpoint, and junction reports. OVITO output visualizes the relaxed
spherocylinders; BPM output converts them into a bonded-particle model
that downstream solvers such as DIRT can consume. Passing
`debug_ovito_path` with no `debug_snapshot_interval` records concise
recipe keyframes. Set an integer interval only when detailed relaxation
frames are needed.

Change `RUN_OVERVIEW` to `True` only when you want to execute the solve.

In [ ]:
if RUN_OVERVIEW:
    # With no snapshot interval, OVITO records recipe milestones only.
    output.mkdir(parents=True, exist_ok=True)
    debug_dump = output / "overview_debug.dump"
    debug_view = output / "overview_debug_view.py"
    debug_session = output / "overview_debug.ovito"
    result = recipe.run(
        settings,
        checkpoint=checkpoint,
        debug_ovito_path=debug_dump,
        debug_ovito_view_script_path=debug_view,
        debug_ovito_session_path=debug_session,
        debug_ovito_coloring="curvature_ratio",
    )

    summary = {
        "converged": result.converged,
        "iterations": result.iterations,
        "max_penetration_um": result.max_penetration * 1.0e6,
        "max_curvature_ratio": result.max_curvature_ratio,
        "active_segments": result.active_segments,
        "splits": result.segment_splits,
        "merges": result.segment_merges,
        "junctions": result.junction_count,
    }
    print(summary)
    print(f"Debug OVITO: {debug_dump} ({result.debug_ovito_frames} frames)")
    print(f"OVITO view recipe: {debug_view}")
    print(f"OVITO session target: {debug_session}")

    # Keep a separate one-frame file for inspecting only the final state.
    result.write_ovito(
        output / "overview_final.dump",
        view_script_path=output / "overview_final_view.py",
        session_path=output / "overview_final.ovito",
        coloring="curvature_ratio",
    )
    # Exact spherocylinders preserve the final active segmentation.
    result.export_bpm(
        output / "overview.data",
        mode="spherocylinders-exact",
        density=1_800.0,
    )

## Where each idea goes next

| Tutorial | Focus |
| --- | --- |
| 02 | Cells, periodic axes, and hard walls |
| 03 | Materials, placed geometry, and rest geometry |
| 04 | Detached fiber collections and metadata |
| 05 | Seeded fiber population generation and directional bias |
| 06 | Insertion, selections, and rigid transforms |
| 07 | Relaxation, contact, mechanics, and CubeCL backends |
| 08 | Adaptive refinement and coarsening |
| 09 | Per-stage solve policies and temporary overrides |
| 10 | Layer movement, target release, and needling |
| 11 | Dynamic compaction and wall control |
| 12 | Explicit junction capture |
| 13 | Checkpoints, continuation, and branching |
| 14 | Results, OVITO visualization, and BPM export |